# Cold-warm trend: load & compute

_Split from `cw_mean_trend.ipynb`. Shared helpers live in `functions.py`._


In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib
from pathlib import Path



In [ ]:
from functions import *  # shared helper functions


In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib
from pathlib import Path



In [ ]:
# omip models - so not access-om2 era5 run

# load percentile data processed in gadi 

mod_labs = ['MRI-ESM2-0', 'FGOALS-f3-L','EC-EARTH3','ACCESS-OM2-JRA55-cycle6_iaf'] # ,'ACCESS-OM2-ERA5'

save_directory = '/Users/isabelaconde/Desktop/data/omip/sst_percentiles/'

file_paths = [f"{save_directory}SST_percentile_{model}_1982-2018_gr.nc" for model in mod_labs]

# some datasets have cftime - change in loop below
import datetime
td = datetime.datetime.strptime("1982-01-01", "%Y-%m-%d")
k = 444
d = pd.date_range(td, periods=k, freq='MS')

dict_percentiles_omip = {}

for i, file in enumerate(file_paths):
    dict_percentiles_omip[mod_labs[i]] = xr.open_dataarray(file).assign_coords({'time': d})


# apply function to each model

percentiles_omip_trend = {}
for i, da in enumerate(mod_labs):
    
    percentiles_omip_trend[da]=find_trend_percentile(dict_percentiles_omip[da].sel(time = slice('1982','2017')))
    percentiles_omip_warm50_mean = (dict_percentiles_omip[da].sel(area = slice(0.5,1)).mean(dim = 'area')).mean(dim = 'time')
    percentiles_omip_cold50_mean = (dict_percentiles_omip[da].sel(area = slice(0,0.5)).mean(dim = 'area')).mean(dim = 'time')
    print('trend and mean calculated for ' + mod_labs[i])

# for i in range(len(mod_labs)):
#     # plot trends of highest and lowest model 
#     percentiles_omip_trend[mod_labs[i]].plot(label = mod_labs[i])  





In [ ]:
# cmip modules 
trend_percentile = xr.open_datatree('/Users/isabelaconde/Desktop/data/dt_cmip_percentiles_trend_1982-2018.nc')

# find mods
# Get ensemble names - yes so you can call on it later
models =['CESM1-CAM5',
 'CanESM2',
 'GFDL-ESM2M',
 'MPI-ESM',
 'ACCESS-ESM1-5',
 'CanESM5',
 'EC-Earth3',
 'IPSL-CM6A-LR',
 'MIROC-ES2L',
 'MIROC6',
 'MPI-ESM1-2-LR',
 'CESM2',
 'GFDL-SPEAR-MED']

data_tree=trend_percentile

mods = {}
for model_name, model_data in data_tree.items():
    en=[]
    for ensemble_name, ensemble_data in data_tree[model_name].items():
        en.extend([str(ensemble_name)])
    
    mods[str(model_name)]=en

    
trend_dt=trend_percentile
model_array=[]
da=[]

for m in range(len(models)):
    sources = mods[models[m]][0:len(mods[models[m]])]
    # da=[] # initialise array for grad values
    for i in range(len(sources)):
        da.extend(trend_dt[models[m]].grad.values)

DA_all = xr.DataArray(np.squeeze(da), dims=['run','area'], coords={'run': np.linspace(0, 1, 519), 'area': np.linspace(0, 1, 100)})


# DA_all= xr.open_dataarray('/Users/isabelaconde/Desktop/data/final_data/da_all_cmip56_runs.nc')

def calculate_confidence_intervals_all(data, n):
    mean = np.empty(n)
    lower = np.empty(n)
    upper = np.empty(n)

    for i in range(data.shape[0]):
        mean[i] = np.mean(data[i])
        lower[i] = scoreatpercentile(data[i],10)
        upper[i] = scoreatpercentile(data[i],90)

    return mean, lower, upper

# Plot for first three models
num=100
data = DA_all.values.T * 10 * 12
mean, lower, upper = calculate_confidence_intervals_all(data, num)
df = pd.DataFrame({'mean': DA_all.mean(dim='run').values * 10 * 12, 
                   'min': DA_all.min(dim='run').values * 10 * 12, 
                   'max': DA_all.max(dim='run').values * 10 * 12, 
                   'lower': lower, 'upper': upper,
                   'area': np.linspace(0, 1, 100)})


In [ ]:
from xhistogram.xarray import histogram

def percentiles_xhist(data): # find percentiles of SST using xhistogram
    hist_T = data.squeeze()
    weights = np.cos(np.deg2rad(hist_T.lat))
    times = hist_T['time']
    const_area=np.r_[0:1:0.01]
    temps = np.r_[12:34:0.5] # define temperatures for binning 
    
    temps_interp = np.empty([len(times),len(const_area)])
    
    for i in np.r_[0:len(times):1]:  
        h=histogram(hist_T[i],bins=temps,weights=weights).cumsum()
        hh = (h - h[0])/(h[-1]-h[0])
        temps_interp[i]=np.interp(const_area,hh,temps[1:])
    
    
    sst_area= xr.DataArray(temps_interp,coords={'time':times,'area':const_area})

    return(sst_area)

def pacific_mask(sst):

    LonsPts_pos = [130,120,110,110,110,260,270,270,277,291,291,130] #-np.ones(13)*180
    LatsPts_pos =[-20,-10,-5,10,20,20,15,15,10,5,-20,-20]
    
    x_pixel_pos = LonsPts_pos
    y_pixel_pos = LatsPts_pos
    
    temp_list = []
    for a, b in zip(x_pixel_pos, y_pixel_pos):
        temp_list.append([a, b])
    
    polygon = np.array(temp_list)
    
    xv, yv = sst.lon.values, sst.lat.values
    lon2d, lat2d = np.meshgrid(xv, yv)
    points = np.hstack((lon2d.reshape((-1,1)), lat2d.reshape((-1,1))))
    path = matplotlib.path.Path(polygon)
    mask = path.contains_points(points)
    mask.shape = lon2d.shape
    mask_bit_pos = mask*np.ones_like(lon2d)

    mask_mods = mask_bit_pos
    
    mask_mods[mask_mods==0] = np.nan
    return mask_mods


def trend_obs_percentile(data): # find trend of percentile data
    time_axis = np.arange(data.sizes['time'])
    trends = np.empty(data.sizes['area'])
    for i in range(data.sizes['area']):
        y = data[:, i].values
        trends[i] = np.polyfit(time_axis, y, 1)[0]  # slope

    return xr.DataArray(trends, coords={'area': data.area}, dims='area', name='sst_trend')


noaa_sst = xr.open_dataset('/Users/isabelaconde/Desktop/data/conde2025/NOAA_sst.mon.mean.nc').sst.sel(time = slice('1982','2017')).sel(lat = slice(-80,80))
noaa_sst_pacific = (pacific_mask(noaa_sst)*noaa_sst).sel(lat=slice(-20,20), lon = slice(100,300))
noaa_sst_percentiles = percentiles_xhist(noaa_sst_pacific)
noaa_warm50_mean = (noaa_sst_percentiles.sel(area = slice(0.5,1)).mean(dim = 'area')).mean(dim = 'time')
noaa_cold50_mean = (noaa_sst_percentiles.sel(area = slice(0,0.5)).mean(dim = 'area')).mean(dim = 'time')
noaa_sst_trend = trend_obs_percentile(noaa_sst_percentiles)



In [ ]:
control_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_control.nc')
era5_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_era5.nc')
perturb_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_merra2_percentile_trend.nc')
# perturb_percent_era5_uas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_percentile_era5_uas_trend.nc')
perturb_percent_era5_uas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/1deg_jra55_iaf_era5_uas_trend_percentile.nc')
perturb_percent_era5_uas_vas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_uas_vas.nc')
perturb_percent_era5_huss_tas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_huss_tas.nc')
perturb_percent_era5_rsds_rlds_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_rsds_rlds.nc')
perturb_percent_era5_winds = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_era5_winds.nc')
perturb_percent_all_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_all_trend.nc')
perturb_percent_winds_mean_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_percentile_trend_uas_vas_mean_trend.nc')


warm50_perturb_uas_vas   = perturb_percent_era5_uas_vas_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_huss_tas  = perturb_percent_era5_huss_tas_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_winds     = perturb_percent_era5_winds.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_all       = perturb_percent_all_trend.sel(area=slice(0.5, 1)).mean(dim="area")
# warm50_cm2 = cm2_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area=slice(0.5, 1)).mean(dim="area")
print("perturbation calculated warm 50 mean")

# cold 50 calc
cold50_jra55 = control_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

cold50_perturb_uas_vas   = perturb_percent_era5_uas_vas_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_huss_tas  = perturb_percent_era5_huss_tas_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_winds     = perturb_percent_era5_winds.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_all       = perturb_percent_all_trend.sel(area=slice(0, 0.5)).mean(dim="area")
# cold50_cm2 = cm2_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold_50_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area=slice(0, 0.5)).mean(dim="area")
print("perturbation calculated cold 50 mean")

In [ ]:
warm50_noaa = noaa_sst_trend.sel(area=slice(0.5, 1)).mean(dim="area")
print("noaa calculated")

warm50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_era5  = era5_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")

cold50_noaa = noaa_sst_trend.sel(area=slice(0, 0.5)).mean(dim="area")
print("noaa calculated")

cold50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")


In [ ]:

p = Path("/Users/isabelaconde/Desktop/data/exp_percentiles")
data = {f.stem: xr.open_dataset(f).to_array().squeeze(drop=True) for f in p.glob("*.nc")}

warm_means = {k: v.sel(area=slice(0.5, 1)).mean(dim=[d for d in ("time", "area") if d in v.dims]) for k, v in data.items()}
cold_means = {k: v.sel(area=slice(0, 0.5)).mean(dim=[d for d in ("time", "area") if d in v.dims]) for k, v in data.items()}


In [ ]:
warm50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_jra55 = control_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

warm50_era5  = era5_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")